# Chimpanzee Re-ID — MegaDescriptor zero-shot pipeline

Wellington Zoo project. This notebook runs the **real** (GPU) version of the pipeline against CZoo/CTai (Freytag et al. 2016) as a stand-in dataset. Everything here mirrors `run_pipeline.py` in the repo — this notebook is a thin, step-by-step wrapper around it for Colab, not a separate implementation.

**Runtime → Change runtime type → GPU**, before running anything below.

## 1. Setup

In [ ]:
!pip install -q -r requirements.txt
!git clone -q https://github.com/cvjena/chimpanzee_faces

## 2. Build manifests

Converts the raw Freytag-format annotations into this project's standard `manifests/*.csv`. Only needs to run once (or again if you re-clone fresh data).

In [ ]:
!python adapters/czoo_to_manifest.py
!python adapters/ctai_to_manifest.py

## 3. Sanity check with `--dry-run` first

This validates manifests, image loading, and the matching/logging logic using fake embeddings — no GPU time spent. A green run here means the only untested part left is MegaDescriptor's actual forward pass, run for real in the next cell.

In [ ]:
!python run_pipeline.py --config config/czoo.yaml --dry-run

## 4. Real run — CZoo

This is the one that actually calls MegaDescriptor. First run will download model weights from the HF hub and will be slow; embeddings are cached to disk afterwards (`cache/embeddings/`), so re-running is fast.

In [ ]:
!python run_pipeline.py --config config/czoo.yaml

## 5. Real run — CTai

Report Section 6: evaluate CZoo and CTai **separately**, don't pool them — CTai is field-quality per Freytag et al., Wellington is expected to resemble CZoo more closely. Compare the two `top_1_accuracy` numbers below against each other, not against a single combined figure.

In [ ]:
!python run_pipeline.py --config config/ctai.yaml

## 6. Inspect the running experiment log

Every run (dry or real) appends one row here. This is the log to paste into the next progress report to supervisors.

In [ ]:
import pandas as pd
pd.read_csv('results/experiment_log.csv')

## 7. Next: Wellington data

Once real photos + keeper-verified identity labels exist:

1. Write/adjust `adapters/wellington_to_manifest.py` for whatever layout the data actually arrives in (folder-per-individual or flat-folder-plus-CSV — both are sketched in that file already).
2. Copy `config/wellington.yaml.template` → `config/wellington.yaml`, fill in `image_root`, and set `precropped` correctly.
3. Run the two cells below — nothing else in this notebook should need editing.

In [ ]:
!python adapters/wellington_to_manifest.py --mode folder --data-root wellington_data --out manifests/wellington_manifest.csv
!python run_pipeline.py --config config/wellington.yaml --dry-run  # check first
!python run_pipeline.py --config config/wellington.yaml            # then the real run